In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import KFold

# Add any additional imports here (however, the task is solvable without using 
# any additional imports)
# import ...

 #### Loading data

In [6]:
# import relevant path
from pathlib import Path
DATA_PATH = Path("../data")



data = pd.read_csv(DATA_PATH / "train.csv")
y = data["y"]
data = data.drop(columns="y")


#### Calculating the average RMSE

In [29]:
def calculate_RMSE(w, X, y):
    """This function takes test data points (X and y), and computes the empirical RMSE of 
    predicting y from X using a linear model with weights w. 

    Parameters
    ----------
    w: array of floats: dim = (13,), optimal parameters of ridge regression 
    X: matrix of floats, dim = (15,13), inputs with 13 features
    y: array of floats, dim = (15,), input labels

    Returns
    ----------
    rmse: float: dim = 1, RMSE value
    """
    rmse = np.sqrt(np.dot(X@w -y, X@w -y) / len(y))

    assert np.isscalar(rmse)
    return rmse


#### Fitting the regressor

In [30]:


def fit(X, y, lam):
    """
    This function receives training data points, then fits the ridge regression on this data
    with regularization hyperparameter lambda. The weights w of the fitted ridge regression
    are returned. 

    Parameters
    ----------
    X: matrix of floats, dim = (135,13), inputs with 13 features
    y: array of floats, dim = (135, ), input labels
    lam: float. lambda parameter, used in regularization term

    Returns
    ----------
    w: array of floats: dim = (13,), optimal parameters of ridge regression
    """
    d = len(X[1])

    
    weights = np.linalg.inv(X.T @ X + lam * np.eye(d)) @ X.T @ y
    
    # TODO: Enter your code here
    assert weights.shape == (13,)
    return weights

#### Performing computation

In [31]:
"""
Main cross-validation loop, implementing 10-fold CV. In every iteration 
(for every train-test split), the RMSE for every lambda is calculated, 
and then averaged over iterations.

Parameters
---------- 
X: matrix of floats, dim = (150, 13), inputs with 13 features
y: array of floats, dim = (150, ), input labels
lambdas: list of floats, len = 5, values of lambda for which ridge regression is fitted and RMSE estimated
n_folds: int, number of folds (pieces in which we split the dataset), parameter K in KFold CV

Compute
----------
avg_RMSE: array of floats: dim = (5,), average RMSE value for every lambda
"""

from sklearn.model_selection import KFold

X = data
# The function calculating the average RMSE
lambdas = [0.1, 1, 10, 100, 200]
n_folds = 10

RMSE_mat = np.zeros((n_folds, len(lambdas)))

# TODO: Enter your code here. Hint: Use functions 'fit' and 'calculate_RMSE' with training and test data
# and fill all entries in the matrix 'RMSE_mat'
l = 0
kf = KFold(n_splits = n_folds, shuffle = True, random_state=42)

for lam in lambdas: 
    
    fold = 0
    for train_i, val_i in kf.split(X):
        X_train, X_val = X.iloc[train_i], X.iloc[val_i]
        y_train, y_val = y.iloc[train_i], y.iloc[val_i]
        
        w_ridge = fit(X_train.to_numpy(), y_train.to_numpy(), lam)
        
        
        RMSE_mat[fold][l] = calculate_RMSE(w_ridge, X_val.to_numpy(), y_val.to_numpy())
        fold += 1
    l+= 1
    print(lam)
    print(RMSE_mat)

avg_RMSE = np.mean(RMSE_mat, axis=0) # avg_RMSE: array of floats: dim = (5,), average RMSE value for every lambda
assert avg_RMSE.shape == (5,)

0.1
[[ 5.92156993  0.          0.          0.          0.        ]
 [ 3.575661    0.          0.          0.          0.        ]
 [ 3.20191332  0.          0.          0.          0.        ]
 [ 5.61385565  0.          0.          0.          0.        ]
 [ 6.05991957  0.          0.          0.          0.        ]
 [ 5.49645433  0.          0.          0.          0.        ]
 [ 4.22945734  0.          0.          0.          0.        ]
 [ 4.15207605  0.          0.          0.          0.        ]
 [10.17907744  0.          0.          0.          0.        ]
 [ 5.40803108  0.          0.          0.          0.        ]]
1
[[ 5.92156993  5.94247752  0.          0.          0.        ]
 [ 3.575661    3.54179062  0.          0.          0.        ]
 [ 3.20191332  3.05277448  0.          0.          0.        ]
 [ 5.61385565  5.6430793   0.          0.          0.        ]
 [ 6.05991957  6.00693616  0.          0.          0.        ]
 [ 5.49645433  5.49312691  0.          0.       

In [33]:
# Save results in the required format
np.savetxt("./results_maurice.csv", avg_RMSE, fmt="%.12f")